# 05 · Geografía: municipios, suelo y overlay
**Responsables:** P3 (municipios) y P4 (suelo) · **Código:** `src/geo.py`, `src/silver_municipios.py`, `src/silver_suelo.py`, `src/gold_suelo.py`
**Salidas:** `silver/municipios.geoparquet`, `silver/capacidad_uso.geoparquet`, `gold/suelo_municipio.parquet`, `gold/suelo_resumen.parquet`

**Qué hace**
1. Reconstruye los polígonos desde el WKT de bronze con el CRS registrado (EPSG:4326).
2. Cuenta geometrías nulas, vacías e inválidas, las repara con `make_valid` y deja solo partes poligonales.
3. **P3:** valida 123 municipios únicos y mide cuánto cambia el área por la simplificación a 75 m frente a `area_km2` oficial.
4. **P4:** valida las clases (III–VIII, ZU, CA) y compara el área calculada con la que reporta el IGAC.
5. **Overlay (P4):** reproyecta ambas capas a **EPSG:9377** (metros), disuelve el suelo por clase (de unos 958 polígonos a unos 8) e intersecta con los municipios. Calcula el % de cada clase sobre el área cubierta y el % sin clasificar.

**Limitaciones que se declaran:** escala 1:100.000 y año 2004. Es capacidad de uso, no fertilidad actual. La clase predominante no implica que todo el municipio tenga ese suelo.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [ ]:
from src.silver_municipios import construir_municipios
pob = pl.read_parquet(SILVER / "poblacion.parquet")
mun = construir_municipios(pob)
mun.head()

In [ ]:
from src.silver_suelo import construir_capacidad_uso
suelo = construir_capacidad_uso()
suelo["clase"].value_counts()

In [ ]:
from src.gold_suelo import composicion_suelo, distribucion_departamental
largo, resumen = composicion_suelo(mun, suelo)
resumen.sort("pct_sin_clasificar", descending=True).head(10)

In [ ]:
from src import graficos
graficos.distribucion_clases(distribucion_departamental(largo).to_pandas())

In [ ]:
pl.read_csv(QUALITY / "metricas_silver.csv").filter(pl.col("fuente").is_in(["municipios", "capacidad_uso"]))